In [2]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np

# Load the raw dataset
df = pd.read_csv('../data/dataset-tickets-multi-lang-4-20k.csv')

# Filter to English only
df_en = df[df['language'] == 'en'].copy()

# Handle Missing Values
df_en.dropna(subset=['answer', 'queue', 'subject', 'body'], inplace=True)

# Since 'id' is missing from the raw CSV, drop duplicates based on the actual ticket text
df_en.drop_duplicates(subset=['subject', 'body', 'answer'], inplace=True) 

# Generate the 'id' column required by the Phase 0/1 database schema
df_en.insert(0, 'id', range(1, len(df_en) + 1))

# Stratified Sample (~8,000 tickets) to maintain class balance
target_size = 8000
if len(df_en) > target_size:
    df_sampled = df_en.groupby('queue', group_keys=False).apply(
        lambda x: x.sample(int(np.rint(target_size * len(x) / len(df_en))), random_state=42)
    )
else:
    df_sampled = df_en

# Clean Text columns by stripping stray whitespace
cols_to_clean = ['subject', 'body', 'answer']
for col in cols_to_clean:
    df_sampled[col] = df_sampled[col].astype(str).str.strip()

# Save to SQLite Warehouse
engine = create_engine('sqlite:///../data/warehouse.db')
df_sampled.to_sql('tickets', engine, index=False, if_exists='replace')

print(f"Warehouse loaded successfully. Total English tickets: {len(df_sampled)}")

Warehouse loaded successfully. Total English tickets: 7999
